# run_chesscog: compare with pretrained chesscog on my board (M4)

Run in a separate environment with chesscog installed (it pins its own torch).
Not executed in CI. Produces `results/<run_name>/chesscog_vs_ours.csv`.

**TODO**
- [ ] implement `chesscog_compare.ipynb: run_chesscog`
- [ ] make `evaluate_model` also write `results/<run>/boards.csv`
      (`session_id, frame_id, fen_true, fen_ours`)

In [ ]:
# !pip install chesscog
# !python -m chesscog.occupancy_classifier.download_model
# !python -m chesscog.piece_classifier.download_model

In [ ]:
%run ./chesscog_compare.ipynb
%run ./config.ipynb
%cd ..
from pathlib import Path
import pandas as pd

cfg = load_config("configs/train.yaml")
RUN = "mixed"   # which of our runs to compare against

In [ ]:
rows = []
for session in cfg.data.test_sessions:
    frames = sorted(Path(cfg.data.crops_root, session, "frames").glob("*.png"))
    fens = run_chesscog(frames)          # TODO M4
    rows += [{"session_id": session, "frame_id": f.stem, "fen_chesscog": fen} for f, fen in zip(frames, fens)]
chesscog_csv = Path("results", RUN, "chesscog.csv")
pd.DataFrame(rows).to_csv(chesscog_csv, index=False)

In [ ]:
df = compare(Path("results", RUN, "boards.csv"), chesscog_csv, Path("results", RUN, "chesscog_vs_ours.csv"))
print("full-board accuracy  ours:", df.correct_ours.mean(), " chesscog:", df.correct_chesscog.mean())
print("mean squares wrong   ours:", df.squares_wrong_ours.mean(), " chesscog:", df.squares_wrong_chesscog.mean())